### import env var and libraries

In [27]:
from openai import OpenAI
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
import gradio as gr

load_dotenv()

client = OpenAI()

### set up Pushover

In [28]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"


In [29]:
#Test pushover
import requests

def send_notification(message: str):
    payload = { "user": pushover_user, "token": pushover_token, "message": message }
    response = requests.post(pushover_url, data=payload)
    return response

In [30]:
send_notification("Hello from VScode!")

<Response [200]>

### describe pushover as an LLM tool

In [31]:
send_notification_function = {
    "name": "send_notification",
    "description": "Sends a pushover notification to the user's phone via the pushover API. Use this to alert the user about important information.",
    "parameters": {
        "type": "object",
        "properties": {
            "message": {
                "type": "string",
                "description": "The message to send in the notification."
             }
            },
        "required": ["message"]
        }
}

### add pushover to the list of tools for LLM

In [32]:
tools = [{"type": "function", "function": send_notification_function}]

### Calling the tool from an LLM

In [36]:
from litellm import completion
client = OpenAI()
response = completion(
    model="gpt-4.1-mini",
    messages = [{"role": "user", "content": "Send me a push notification to my phone of a fun fact about software engineering someone who is \
    trying to land an intern role would like to hear (context: I am a first year Data Scienc major in SJSU)"}],
    tools=tools,
    tool_choice="auto"
)
message = response.choices[0].message

print(message)

Message(content=None, role='assistant', tool_calls=[ChatCompletionMessageToolCall(function=Function(arguments='{"message":"Fun Fact: The first computer programmer was Ada Lovelace in the 19th century! For software engineering interns, mastering version control like Git is essential—it’s widely used in collaboration and job settings. Keep practicing coding challenges and building small projects to impress SJSU recruiters!"}', name='send_notification'), id='call_Ipxygyf3S66fFTknQFKtfwR6', type='function')], function_call=None, provider_specific_fields={'refusal': None}, annotations=[])


In [38]:
if message.tool_calls: 
    tool_call = message.tool_calls[0]
    import json
    args = json.loads(tool_call.function.arguments)
    
    send_notification(args['message'])
    print("Notification sent!")
    
else:
    print(message.content)


Notification sent!


### LLM tool calling function (unified)

In [48]:
from litellm import completion
import json

client = OpenAI()

send_notification_function = {
    "name": "send_notification",
    "description": "Sends a pushover notification to the user's phone via the pushover API. Use this to alert the user about important information.",
    "parameters": {
        "type": "object",
        "properties": {
            "message": {
                "type": "string",
                "description": "The message to send in the notification."
             }
            },
        "required": ["message"]
        }
}

tools = [{"type": "function", "function": send_notification_function}]

prompt = [{"role": "user", "content": "Send me a push notification to my phone of a fun fact about software/ai engineering \
            for someone who is trying to land an intern role in the next 3-6 months. keep it 1-2.5 sentances \
            (context: I am a first year Data Scienc major in SJSU)"}]

response = completion(
    model="gpt-4.1-mini",
    messages = prompt,
    tools=tools,
    temperature= 1.1,
    tool_choice="auto"
)
message = response.choices[0].message

if message.tool_calls: 
    tool_call = message.tool_calls[0]
    args = json.loads(tool_call.function.arguments)
    
    send_notification(args['message'])
    print("Notification sent!")
    
else:
    print(message.content)


Notification sent!
